## Extensive finite difference tests for the inflatable sheet's energy, digging into the individual energy density terms/Cauchy-Green tensors.

In [ ]:
import sys
sys.path.append('..')
import inflation, numpy as np, importlib, fd_validation, mesh

In [ ]:
m = mesh.Mesh('../examples/single_tri.obj')
isheet = inflation.InflatableSheet(m)

In [ ]:
Pressure = inflation.InflatableSheet.EnergyType.Pressure
Elastic  = inflation.InflatableSheet.EnergyType.Elastic
Full     = inflation.InflatableSheet.EnergyType.Full
isheet.pressure = 1

In [ ]:
isheet.tensionStateHistogram()

In [ ]:
# perturbation putting the single triangle example in a partial tension state
eval_perturb  = np.array([-0.25989305, -0.36850698, -0.9096837 , -0.42756298, -0.2256421,  -0.83363732,  0.51394147, -0.60510383, -0.43363783])

In [ ]:
fd_perturb = np.random.uniform(low=-1.0, high=1.0, size=isheet.numVars())

In [ ]:
xorig = isheet.getVars()
xeval = isheet.getVars() + 1e-3 * eval_perturb

In [ ]:
# Set current point as "x_eval" and verify that tension state doesn't change in the surrounding neighborhood 
isheet.setVars(xeval + 1e-6 * fd_perturb)
print(isheet.tensionStateHistogram())
isheet.setVars(xeval - 1e-6 * fd_perturb)
print(isheet.tensionStateHistogram())
isheet.setVars(xeval)
print(isheet.tensionStateHistogram())

In [ ]:
# FD validations for the energy density derivatives wrt C
def randSym():
    a = np.random.uniform(low=-1, high=1, size=(2, 2))
    a[0, 1] = a[1, 0] = 0.5 * (a[0, 1] + a[1, 0])
    return a

def EDensity(C):
    result = inflation.OptionalTensionFieldEnergy(C)
    result.useTensionField = True
    return result
def validateEnergyDensity_Grad(C, fd_eps = 1e-6, perturb = None):
    if perturb is None: perturb = randSym()
    ted = EDensity(C)
    eplus = EDensity(C + fd_eps * perturb).energy()
    emins = EDensity(C - fd_eps * perturb).energy()
    fd_delta_e = (eplus - emins) / (2 * fd_eps)
    an_delta_e = ted.denergy(perturb)
    return np.abs((fd_delta_e - an_delta_e))
def validateEnergyDensity_Hess(C, fd_eps = 1e-6, perturb_a = None, perturb_b = None):
    if perturb_a is None: perturb_a = randSym()
    if perturb_b is None: perturb_b = randSym()
    ted = EDensity(C)
    deplus = EDensity(C + fd_eps * perturb_b).denergy(perturb_a)
    demins = EDensity(C - fd_eps * perturb_b).denergy(perturb_a)
    fd_delta_de = (deplus - demins) / (2 * fd_eps)
    an_delta_de = ted.d2energy(perturb_a, perturb_b)
    return np.abs((fd_delta_de - an_delta_de))

In [ ]:
test_perturb_a, test_perturb_b = randSym(), randSym()
print(np.max([validateEnergyDensity_Grad(C, 1e-6, test_perturb_a) for C in isheet.cauchyGreenDeformationTensors()]), "\n",
      np.max([validateEnergyDensity_Hess(C, 1e-6, test_perturb_a, test_perturb_b) for C in isheet.cauchyGreenDeformationTensors()]))

In [ ]:
# FD validation of C's derivatives with respect to the sheet variables
def fd_dC(sheetTriIdx, varIdx, fd_eps = 1e-7):
    x_curr = isheet.getVars()
    x_perturb = x_curr.copy()
    x_perturb[varIdx] += fd_eps
    isheet.setVars(x_perturb)
    C_plus  = isheet.cauchyGreenDeformationTensors()[sheetTriIdx]
    x_perturb[varIdx] -= 2 * fd_eps
    isheet.setVars(x_perturb)
    C_minus = isheet.cauchyGreenDeformationTensors()[sheetTriIdx]
    isheet.setVars(x_curr)
    return (C_plus - C_minus) / (2 * fd_eps)

def fd_d2C(sheetTriIdx, varIdxA, varIdxB, fd_eps = 1e-7):
    x_curr = isheet.getVars()
    x_perturb = x_curr.copy()
    x_perturb[varIdxB] += fd_eps
    isheet.setVars(x_perturb)
    dC_plus  = fd_dC(sheetTriIdx, varIdxA, fd_eps)
    x_perturb[varIdxB] -= 2 * fd_eps
    isheet.setVars(x_perturb)
    dC_minus  = fd_dC(sheetTriIdx, varIdxA, fd_eps)
    isheet.setVars(x_curr)
    return (dC_plus - dC_minus) / (2 * fd_eps)

In [ ]:
# Semi-ananalytical derivatives of full Elastic energy using finite difference derivatives of C but analytic derivatives of the energy density
areas = isheet.undeformedAreas()
def fdC_based_grad(varIdx):
    return np.dot(areas, np.array([ted.denergy(fd_dC(tIdx, varIdx)) for tIdx, ted in enumerate(isheet.triEnergyDensities())]))

def fdC_based_hess(varIdxA, varIdxB):
    return np.dot(areas, np.array([ted.denergy(fd_d2C(tIdx, varIdxA, varIdxB)) +
                                        ted.d2energy(fd_dC(tIdx, varIdxA),
                                                     fd_dC(tIdx, varIdxB))
                for tIdx, ted in enumerate(isheet.triEnergyDensities())]))

In [ ]:
# Finite difference of the semi-analytic gradient
def fd_fdC_based_grad(varIdxA, varIdxB, fd_eps = 1e-7):
    x_curr = isheet.getVars()
    x_perturb = x_curr.copy()
    x_perturb[varIdxB] += fd_eps
    isheet.setVars(x_perturb)
    gradA_plus = fdC_based_grad(varIdxA)
    x_perturb[varIdxB] -= 2 * fd_eps
    isheet.setVars(x_perturb)
    gradA_minus = fdC_based_grad(varIdxA)
    isheet.setVars(x_curr)
    return (gradA_plus - gradA_minus) / (2 * fd_eps)

In [ ]:
import sparsity_analysis
sparsity_analysis.to_csc(isheet.hessian(Elastic))[0, 0], fd_fdC_based_grad(0, 0)

In [ ]:
def basisVector(n, i):
    e_i = np.zeros(n)
    e_i[i] = 1.0
    return e_i

In [ ]:
H_err = fd_validation.validateHessian(isheet, fd_eps=1e-7, xeval=isheet.getVars(), perturb=basisVector(isheet.numVars(), 0), etype=Elastic)
fd_delta_grad, an_delta_grad = H_err[1:]
H_err

In [ ]:
fd_validation.validateHessian(isheet, xeval=isheet.getVars(), fd_eps=1e-7,
                              etype=Elastic)